# 06 · Resource benchmark — §4, §6

Latency and peak memory depend on the machine, but not on trained weights. So every architecture is built fresh and profiled back to back here — no checkpoints needed, and every row of the §6 table is measured on the same hardware, whoever trained which model where. A local machine is recommended: its CPU model is stable and can be named in the report.

| | |
|---|---|
| **Owner** | Member 1 |
| **Reads** | each run's experiment config (architecture only) and `history.json` (epoch times) |
| **Writes (official)** | `results/metrics/<run_id>/resources.json` for all six runs |
| **Run it** | once, after 03–05's official runs are committed — in **one session, on one CPU** |

In [8]:
# ── Cell 1 · Colab bootstrap ─ does nothing when run locally ───────────────
# On a Colab runtime (in the browser, or from VS Code via the Colab extension) it
# clones REPO at BRANCH, installs it, and on every re-run pulls what you pushed since.
# Locally the package is already installed, so this is skipped. Guide: COLAB.md
import importlib.util
import sys

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    import os
    import subprocess

    def _secret(name, default):  # your Colab Secret if you set one, else the default
        try:
            from google.colab import userdata

            return userdata.get(name) or default
        except Exception:  # not set, or unreadable here (VS Code cannot read Secrets)
            return default

    # The code to run is REPO @ BRANCH. Set your own fork / branch once, in Colab Secrets,
    # as A03_REPO / A03_BRANCH. In VS Code, edit the defaults below for the session and
    # put them back before committing. See COLAB.md.
    REPO = _secret("A03_REPO", "ThejithaR/EN3150-Assignment-03-CNN")
    BRANCH = _secret("A03_BRANCH", "main")
    REPO_DIR = "/content/en3150-a03"

    def _git(*args):  # never prompts: a failure shows as an error, not a hang
        subprocess.run(["git", *args], check=True, env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})

    url = f"https://github.com/{REPO}.git"
    if not os.path.exists(REPO_DIR):
        _git("clone", "--quiet", "--branch", BRANCH, url, REPO_DIR)
    else:  # re-running this cell picks up everything pushed since
        _git("-C", REPO_DIR, "remote", "set-url", "origin", url)
        _git("-C", REPO_DIR, "fetch", "--quiet", "--prune", "origin")
        _git("-C", REPO_DIR, "checkout", "--quiet", "-B", BRANCH, f"origin/{BRANCH}")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", REPO_DIR], check=True)
    if f"{REPO_DIR}/src" not in sys.path:
        sys.path.insert(0, f"{REPO_DIR}/src")  # an editable install is only seen after a restart
    os.chdir(REPO_DIR)
    _git("log", "--oneline", "-1")  # prints the exact commit this run uses

In [9]:
# ── Cell 2 · Setup ─────────────────────────────────────────────────────────
%load_ext autoreload
%autoreload 2

# MODE decides how much runs and whether anything is written:
#   "synthetic"  generated data, 2 epochs, seconds         - before EuroSAT exists
#   "debug"      5% of EuroSAT, 2 epochs                   - find bugs step by step
#   "official"   the full run - the ONLY mode that writes  - then Restart & Run All
MODE = "synthetic"

from edgecnn.config import load_config
from edgecnn.contracts import paths, schema
import dataclasses
import platform

import pandas as pd

from edgecnn.evaluation import profile_all
from edgecnn.utils import describe_device, resolve_device

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Runs to profile

Every run that appears in a report table.

In [10]:
tables = load_config("configs/experiments/model_b__adam.yaml").section("reporting")["tables"]
RUN_IDS = sorted({run for table in tables for run in table["runs"]})
RUN_IDS

['mobilenet_v2__adam__seed42',
 'model_a__adam__seed42',
 'model_b__adam__seed42',
 'model_b__sgd__seed42',
 'model_b__sgd_momentum__seed42',
 'squeezenet1_1__adam__seed42']

## Hardware

This is the machine the §6 latency column describes.

In [11]:
print(platform.processor() or platform.machine())
print(describe_device(resolve_device("cpu")))

Intel64 Family 6 Model 141 Stepping 1, GenuineIntel
cpu (11th Gen Intel Core i5-11400H @ 2.70GHz)


## Profile every architecture — one session

For each architecture, built fresh with the same settings its training used:

- **size** — the saved weights, in KB (1 KB = 1,024 bytes);
- **MACs** — multiply-accumulates for one image, counting convolution and fully-connected layers;
- **latency** — the mean of 100 timed forward passes after 10 untimed ones, one image at a time, on **one CPU thread**, as a small edge device would run it;
- **peak memory** — the weights plus the largest set of intermediate results alive at the same time, worked out from the model's graph, in MB (1 MB = 1,048,576 bytes).

The three Model B runs share one architecture, so they share one measurement; only their epoch times, read from each run's `history.json`, differ. Outside official mode, a model that isn't implemented yet is skipped with a warning.

In [12]:
profiles = profile_all(RUN_IDS, mode=MODE)

Skipped mobilenet_v2__adam__seed42: its model is not implemented yet (Member 4: implement MobileNetV2 fine-tuning)
Skipped squeezenet1_1__adam__seed42: its model is not implemented yet (Member 4: implement SqueezeNet fine-tuning)


## Results

In [13]:
pd.DataFrame([dataclasses.asdict(p) for p in profiles.values()]).set_index("run_id")

,model_name,trainable_params,total_params,model_size_kb,macs,mean_epoch_time_s,inference_latency_ms,peak_mem_mb,device
run_id,,,,,,,,,
model_a__adam__seed42,model_a,1143466,1143466,4475.157227,42337536,0.0,3.572387,5.363708,cpu (11th Gen Intel Core i5-11400H @ 2.70GHz)
model_b__adam__seed42,model_b,71818,71818,303.932617,12989568,0.0,3.252390,1.280624,cpu (11th Gen Intel Core i5-11400H @ 2.70GHz)
model_b__sgd__seed42,model_b,71818,71818,303.932617,12989568,0.0,3.252390,1.280624,cpu (11th Gen Intel Core i5-11400H @ 2.70GHz)
model_b__sgd_momentum__seed42,model_b,71818,71818,303.932617,12989568,0.0,3.252390,1.280624,cpu (11th Gen Intel Core i5-11400H @ 2.70GHz)


## Official-run checklist

- [ ] `MODE = "official"` in the setup cell
- [ ] Kernel → **Restart & Run All** finished without errors
- [ ] every output is what you expect — no stray warnings or giant prints
- [ ] 03, 04 and 05 official results were committed before this run
- [ ] ran on CPU, in one session, on one machine — named in the report
- [ ] `pytest tests/contracts` is green
- [ ] commit this notebook **with its outputs**, together with the files it wrote

In [14]:
# ── Colab only: send this run's results back to GitHub ─────────────────────
# Does nothing locally, or in synthetic / debug mode. Guide: COLAB.md
if IN_COLAB and MODE == "official":
    from edgecnn.utils import push_results

    push_results(
        [paths.metrics_dir(run_id) / "resources.json" for run_id in RUN_IDS],
        "Add resource profiles (Colab)",
    )